# 5 прикладів кошиків
Ті самі 5 тестових кошиків даємо всім моделям і порівнюємо рекомендації.

Як обираємо кошики:
1. **найкращий**: кошик з 6+ товарів, де моделі разом вгадали найбільше прихованих товарів
2. **провальний**: 6+ товарів, усі товари кошика є в каталозі, але жодна модель нічого не вгадала
3. **випадкові** (3 шт.): кошики, де хоча б одна модель щось вгадала

Позначки в рекомендаціях:
- **✓** вгадано (товар є в прихованій частині)
- **≈** та сама полиця (aisle), що й товар з видимої частини, можливий замінник
- **★** товар з топ-100 популярних

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from data.loading import load_baskets, load_products
from evaluation.metrics import split_baskets
from models.catalog import build_catalog, product_counts
from models.recommend import load_models, recommend

SEED = 42
K = 10
MODELS = load_models()

train = load_baskets("train")
catalog = build_catalog(train)
top100 = set(product_counts(train).head(100).index)
test_splits = split_baskets(load_baskets("test"))

products = load_products().set_index("product_id")
aisle = products["aisle"].to_dict()
name = products["product_name"].to_dict()
MODELS

## Скільки вгадала кожна модель у кожному тестовому кошику

In [ ]:
rows = []
for order_id, visible, hidden in zip(test_splits["order_id"], test_splits["visible"], test_splits["hidden"]):
    row = {"order_id": order_id, "size": len(visible) + len(hidden),
           "all_in_catalog": all(p in catalog for p in visible + hidden)}
    for model in MODELS:
        recs = recommend(visible, model, K)["product_id"]
        row[model] = len(set(recs) & set(hidden))
    rows.append(row)

hits = pd.DataFrame(rows).set_index("order_id")
hits.describe().round(2)

## Обираємо 5 кошиків

In [ ]:
n_hits = hits[MODELS].sum(axis=1)
big = hits["size"] >= 6

best = n_hits[big].idxmax()
failed = hits[big & hits["all_in_catalog"] & (n_hits == 0)].sample(1, random_state=SEED).index[0]
lucky = hits[(n_hits > 0) & (hits.index != best)].sample(3, random_state=SEED).index.tolist()

examples = {"найкращий": best, "провальний": failed,
            "випадковий 1": lucky[0], "випадковий 2": lucky[1], "випадковий 3": lucky[2]}
hits.loc[list(examples.values())]

In [ ]:
def show_example(order_id):
    split = test_splits.set_index("order_id").loc[order_id]
    visible, hidden = split["visible"], split["hidden"]
    visible_aisles = {aisle[p] for p in visible}

    basket = pd.DataFrame({"видима частина": pd.Series([name[p] for p in visible]),
                           "прихована частина": pd.Series([name[p] for p in hidden])}).fillna("")
    display(basket)

    table = {}
    for model in MODELS:
        marked = []
        for p in recommend(visible, model, K)["product_id"]:
            marks = ("✓" if p in hidden else "") + ("≈" if aisle[p] in visible_aisles else "") + ("★" if p in top100 else "")
            marked.append(f"{marks} {name[p]}".strip())
        table[model] = marked
    display(pd.DataFrame(table, index=range(1, K + 1)))

## Кошик: найкращий

In [5]:
show_example(examples["найкращий"])

,видима частина,прихована частина
0,Organic Lite Coconut Milk,Large Lemon
1,Organic Garbanzo Beans,Limes
2,Organic Low Sodium Chicken Cooking Stock,Sliced Shiitake Mushrooms
3,Organic No Salt Added Diced Tomatoes,Red Vine Tomato
4,Ezekiel 4:9 Bread Organic Sprouted Whole Grain,Organic Zucchini
5,Feta Cheese Crumbles,Organic Avocado
6,Asparagus,Banana
7,Broccoli Crown,Organic Strawberries
8,Mache Rosettes French Salad Mix,Honeycrisp Apple
9,Organic Chopped Garlic,Organic SprouTofu Savory Baked Tofu


,popularity,conditional,cosine,ease
1,✓★ Banana,✓★ Banana,≈★ Organic Baby Spinach,≈★ Organic Baby Spinach
2,★ Bag of Organic Bananas,≈★ Organic Baby Spinach,✓★ Limes,✓★ Limes
3,✓★ Organic Strawberries,★ Bag of Organic Bananas,✓★ Large Lemon,✓★ Large Lemon
4,≈★ Organic Baby Spinach,★ Organic Hass Avocado,✓★ Banana,✓★ Organic Avocado
5,★ Organic Hass Avocado,✓★ Limes,✓★ Organic Avocado,✓★ Banana
6,✓★ Organic Avocado,✓★ Organic Strawberries,≈★ Organic Garlic,≈★ Organic Garlic
7,✓★ Large Lemon,✓★ Large Lemon,★ Organic Hass Avocado,★ Organic Hass Avocado
8,★ Strawberries,✓★ Organic Avocado,≈★ Organic Yellow Onion,≈★ Yellow Onions
9,✓★ Limes,≈★ Organic Garlic,★ Organic Cilantro,★ Bunched Cilantro
10,★ Organic Whole Milk,≈★ Organic Yellow Onion,✓≈★ Organic Zucchini,≈★ Cucumber Kirby


## Кошик: провальний

In [6]:
show_example(examples["провальний"])

,видима частина,прихована частина
0,Sriracha Chili Sauce,"Miso, Organic, Mellow, White"
1,Pad Thai Brown Rice Noodles,Peanut Butter Cereal
2,Organic SprouTofu Super Firm Tofu,Roasted Salted Peanuts
3,Organic Mung-Bean Sprouts,Sea Salt & Vinegar Potato
4,Organic Tomato Paste,Diet Ginger Ale All Natural Soda
5,Limes,Organic Peanut Butter Granola


,popularity,conditional,cosine,ease
1,≈★ Banana,≈★ Banana,★ Organic Garlic,≈★ Large Lemon
2,≈★ Bag of Organic Bananas,≈★ Bag of Organic Bananas,★ Organic Cilantro,★ Organic Cilantro
3,≈★ Organic Strawberries,≈★ Organic Baby Spinach,≈★ Large Lemon,★ Organic Garlic
4,≈★ Organic Baby Spinach,≈★ Organic Hass Avocado,★ Organic Yellow Onion,≈★ Organic Lemon
5,≈★ Organic Hass Avocado,★ Organic Garlic,≈★ Organic Baby Spinach,≈★ Banana
6,≈★ Organic Avocado,★ Organic Yellow Onion,≈★ Banana,★ Bunched Cilantro
7,≈★ Large Lemon,≈★ Organic Strawberries,≈★ Organic Hass Avocado,★ Organic Yellow Onion
8,≈★ Strawberries,≈★ Organic Lemon,≈★ Organic Lemon,≈★ Organic Avocado
9,★ Organic Whole Milk,★ Carrots,≈★ Bag of Organic Bananas,★ Jalapeno Peppers
10,≈★ Organic Raspberries,★ Organic Cilantro,★ Carrots,★ Yellow Onions


## Кошик: випадковий 1

In [7]:
show_example(examples["випадковий 1"])

,видима частина,прихована частина
0,Banana,Annie Peely Fruit 4.5 Z
1,Organic Whole Milk,Soft Potato Bread
2,Cantaloupe,Organic Baby Spinach
3,Organic Strawberries,Organic Grape Tomatoes
4,Organic Avocado,Pure Sparkling Water
5,Sourdough Bread,Limes
6,Organic Peeled Whole Baby Carrots,Organic Red Bell Pepper
7,Honeycrisp Apple,Yellow Bell Pepper
8,Uncured Genoa Salami,Organic Strawberry Fruit Spread
9,Total 0% Nonfat Plain Greek Yogurt,Triple Creme Goat Brie


,popularity,conditional,cosine,ease
1,≈★ Bag of Organic Bananas,≈★ Bag of Organic Bananas,✓≈★ Organic Baby Spinach,✓≈★ Organic Baby Spinach
2,✓≈★ Organic Baby Spinach,✓≈★ Organic Baby Spinach,≈★ Large Lemon,≈★ Large Lemon
3,≈★ Organic Hass Avocado,≈★ Organic Hass Avocado,≈★ Bag of Organic Bananas,≈★ Bag of Organic Bananas
4,≈★ Large Lemon,≈★ Large Lemon,✓≈★ Limes,★ Cucumber Kirby
5,≈★ Strawberries,✓≈★ Limes,★ Cucumber Kirby,≈★ Seedless Red Grapes
6,✓≈★ Limes,≈★ Organic Raspberries,≈★ Organic Raspberries,≈★ Organic Blueberries
7,≈★ Organic Raspberries,★ Cucumber Kirby,≈★ Organic Hass Avocado,≈★ Organic Raspberries
8,★ Organic Yellow Onion,≈★ Strawberries,✓≈★ Organic Grape Tomatoes,≈★ Strawberries
9,★ Organic Garlic,★ Organic Zucchini,≈★ Seedless Red Grapes,✓≈★ Limes
10,★ Organic Zucchini,★ Organic Garlic,★ Organic Whole String Cheese,★ Organic Whole String Cheese


## Кошик: випадковий 2

In [8]:
show_example(examples["випадковий 2"])

,видима частина,прихована частина
0,Grapefruit Sparkling Water,Chocolate Sea Salt
1,Blueberry Protein Bar,Coconut Chocolate Bar


,popularity,conditional,cosine,ease
1,★ Banana,★ Bag of Organic Bananas,Protein Bar Maple Sea Salt,≈ Lemon Sparkling Water
2,★ Bag of Organic Bananas,✓≈ Chocolate Sea Salt,≈ Lemon Sparkling Water,≈ Passionfruit Sparkling Water
3,★ Organic Strawberries,★ Organic Hass Avocado,✓≈ Chocolate Sea Salt,≈ Coconut Sparkling Water
4,★ Organic Baby Spinach,★ Banana,Natural Lemon Flavored Sparkling Water,★ Banana
5,★ Organic Hass Avocado,≈ Lemon Sparkling Water,≈ Coconut Sparkling Water,≈ Orange Sparkling Water
6,★ Organic Avocado,≈ Peanut Butter Egg White Protein Bar,≈ Peach-Pear Sparkling Water,≈ Peach-Pear Sparkling Water
7,★ Large Lemon,Protein Bar Maple Sea Salt,≈ Passionfruit Sparkling Water,Natural Lemon Flavored Sparkling Water
8,★ Strawberries,★ Organic Zucchini,≈ Peanut Butter Egg White Protein Bar,≈ Cran-Raspberry Flavored Sparkling Water
9,★ Limes,✓≈ Coconut Chocolate Bar,≈ Cran-Raspberry Flavored Sparkling Water,★ Bag of Organic Bananas
10,★ Organic Whole Milk,★ Organic Garlic,✓≈ Coconut Chocolate Bar,≈ Blackberry Cucumber Sparkling Water


## Кошик: випадковий 3

In [9]:
show_example(examples["випадковий 3"])

,видима частина,прихована частина
0,Organic Hass Avocado,Organic Baby Spinach
1,"The \""World's Best\"" Veggie Burger",Organic Baby Bella Mushrooms
2,Organic Original Tofu Veggie Burger,Organic Summer Strawberry Bunny Fruit Snacks
3,Gluten Free Whole Grain Bread,Vanilla Bean Chia Pod


,popularity,conditional,cosine,ease
1,≈★ Banana,≈★ Bag of Organic Bananas,≈★ Bag of Organic Bananas,≈★ Bag of Organic Bananas
2,≈★ Bag of Organic Bananas,✓★ Organic Baby Spinach,≈★ Organic Strawberries,✓★ Organic Baby Spinach
3,≈★ Organic Strawberries,≈★ Organic Strawberries,✓★ Organic Baby Spinach,≈★ Organic Strawberries
4,✓★ Organic Baby Spinach,≈★ Banana,★ Organic Cucumber,≈★ Organic Lemon
5,≈★ Organic Avocado,★ Carrots,★ Organic Raspberries,★ Organic Raspberries
6,≈★ Large Lemon,★ Organic Cucumber,≈★ Organic Lemon,★ Organic Cucumber
7,≈★ Strawberries,★ Organic Raspberries,★ Organic Yellow Onion,★ Organic Tomato Cluster
8,≈★ Limes,★ Organic Yellow Onion,≈★ Limes,≈★ Banana
9,★ Organic Whole Milk,≈★ Limes,★ Organic Zucchini,★ Organic Yellow Onion
10,★ Organic Raspberries,★ Organic Zucchini,★ Organic Tomato Cluster,≈★ Apple Honeycrisp Organic


## Частка рекомендацій з тієї ж полиці і з топ-100 (2000 випадкових тестових кошиків)

In [10]:
sample = test_splits.sample(2000, random_state=SEED)
summary = {}
for model in MODELS:
    same_aisle = popular = hit = total = 0
    for visible, hidden in zip(sample["visible"], sample["hidden"]):
        visible_aisles = {aisle[p] for p in visible}
        for p in recommend(visible, model, K)["product_id"]:
            same_aisle += aisle[p] in visible_aisles
            popular += p in top100
            hit += p in hidden
            total += 1
    summary[model] = {"вгадано": hit / total, "та сама полиця (≈)": same_aisle / total, "топ-100 (★)": popular / total}

pd.DataFrame(summary).T.round(3)

,вгадано,та сама полиця (≈),топ-100 (★)
popularity,0.032,0.338,1.000
conditional,0.040,0.394,0.900
cosine,0.041,0.481,0.749
ease,0.041,0.455,0.793
